In [21]:
pip install --no-cache-dir pyarrow==17.0.0 datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.24.1 requires packaging<24,>=14.1, but you have packaging 24.2 which is incompatible.
streamlit 1.24.1 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.
streamlit 1.24.1 requires rich<14,>=10.11.0, but you have rich 14.1.0 which is incompatible.



INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
   -- ------------------------------------- 1.8/25.2 MB 12.6 MB/s eta 0:00:02
   ------- -------------------------------- 5.0/25.2 MB 13.7 MB/s eta 0:00:02
   ------------ --------------------------- 8.1/25.2 MB 14.0 MB/s eta 0:00:02
   ----------------- ---------------------- 11.3/25.2 MB 14.4 MB/s eta 0:00:01
   ---------------------- ----------------- 13.9/25.2 MB 14.3 MB/s eta 0:00:01
   --------------------------- ------------ 17.0/25.2 MB 14.3 MB/s eta 0:00:01
   ------------------------------- -------- 19.9/25.2 MB 14.3 MB/s eta 0:00:01
   ------------------------------------- -- 23.3/25.2 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 25.2/25.2 MB 14.6 MB/s  0:00:01

   ---------------------------------------- 0/2 [pyarrow]
   -----------------

In [22]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch.utils.data import DataLoader

# Transformers
from transformers import DebertaV2ForSequenceClassification, DebertaV2Tokenizer
from transformers import DataCollatorWithPadding, get_scheduler

# Scikit-learn 
from sklearn.metrics import accuracy_score

<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


ImportError: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _dataset: The specified procedure could not be found.)

In [18]:
%pip install --upgrade --force-reinstall pyarrow

# Random seed
random_seed = 42


# Define a dictionary to specify the paths for the train and test sets
data_files = {
    'train': PATH + 'train.csv',
    'test': PATH + 'test.csv'    
}

from datasets import load_dataset

# Load the dataset 
nlp_dataset = load_dataset('csv', data_files=data_files)

nlp_dataset

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.24.1 requires packaging<24,>=14.1, but you have packaging 24.2 which is incompatible.
streamlit 1.24.1 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.
streamlit 1.24.1 requires rich<14,>=10.11.0, but you have rich 14.1.0 which is incompatible.


  Using cached pyarrow-21.0.0-cp311-cp311-win_amd64.whl.metadata (3.4 kB)
Using cached pyarrow-21.0.0-cp311-cp311-win_amd64.whl (26.2 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Uninstalling pyarrow-21.0.0:
      Successfully uninstalled pyarrow-21.0.0
Note: you may need to restart the kernel to use updated packages.


ImportError: The pyarrow installation is not built with support for the Parquet file format (DLL load failed while importing _parquet: The specified module could not be found.)

In [ ]:
# Extract the text for each label
text_label_0 = ' '.join(nlp_dataset['train'].filter(lambda x: x['target'] == 0)['text'])
text_label_1 = ' '.join(nlp_dataset['train'].filter(lambda x: x['target'] == 1)['text'])

# Word cloud for label 0 (Non-Disaster)
wordcloud_0 = WordCloud(
    background_color="black",
    width=900,
    height=830,
    max_words=200,
    colormap="Blues",
    contour_color='white',
    contour_width=2,
    min_font_size=16,
    max_font_size=100,
    prefer_horizontal=0.9,
    random_state=random_seed
).generate(text_label_0)

# Word cloud for label 1 (Disaster)
wordcloud_1 = WordCloud(
    background_color="black",
    width=900,
    height=830,
    max_words=200,
    colormap="Reds",
    contour_color='white',
    contour_width=2,
    min_font_size=16,
    max_font_size=100,
    prefer_horizontal=0.9,
    random_state=random_seed
).generate(text_label_1)

# Create a subplot for both word clouds
plt.figure(figsize=(20, 18))

# Plot word clouds
plt.subplot(1, 2, 1)
plt.imshow(wordcloud_0, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Non-Disaster', fontsize=20, fontweight='bold', color='#333333')

plt.subplot(1, 2, 2)
plt.imshow(wordcloud_1, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Disaster', fontsize=20, fontweight='bold', color='#333333')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Tokenization
checkpoint = 'microsoft/deberta-v3-large' 
tokenizer = DebertaV2Tokenizer.from_pretrained(checkpoint)

def tokenize_function(dataset):
    return tokenizer(dataset['text'], truncation=True,
                     padding='max_length', max_length=128)

tokenized_datasets = nlp_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Prepare datasets
tokenized_datasets = tokenized_datasets.remove_columns(["id", "keyword", "location", "text"])
tokenized_datasets = tokenized_datasets.rename_column("target", "labels")
tokenized_datasets.set_format("torch")

# Split datasets
tokenized_datasets_clean = tokenized_datasets["train"].train_test_split(train_size=0.9, seed=random_seed)
tokenized_datasets_clean["validation"] = tokenized_datasets_clean.pop("test")
tokenized_datasets_clean["test"] = tokenized_datasets["test"]

# Create train and eval data loaders
train_loader = DataLoader(tokenized_datasets_clean["train"], shuffle=True, batch_size=8, collate_fn=data_collator)
val_loader = DataLoader(tokenized_datasets_clean["validation"], batch_size=8, collate_fn=data_collator)

In [ ]:
# Set up the device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

id2label = {0: "Non-Disaster", 1: "Disaster"}
label2id = {"Non-Disaster": 0, "Disaster": 1}

# Load model for sequence classification
model = DebertaV2ForSequenceClassification.from_pretrained(checkpoint, 
                                                           num_labels=2,
                                                           id2label=id2label, 
                                                           label2id=label2id).to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00000077)

# Number of epochs
epochs = 4
num_training_steps = epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,

In [ ]:
# Training function
def train(loader):
    model.train()  # Set the model to training mode
    total_loss = total_samples = total_correct = 0
    y_true = []
    y_pred = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}  # Move to device (GPU/CPU)
        
        optimizer.zero_grad()  # Clear previous gradients
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass and optimization
        loss.backward()
        
        optimizer.step()

        total_loss += loss.item() * batch['input_ids'].size(0)  # Accumulate loss
        total_samples += batch['input_ids'].size(0)

        y_true.extend(batch['labels'].cpu().numpy())
        y_pred.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())

        # Calculate accuracy for this batch
        correct = (torch.argmax(outputs.logits, dim=-1) == batch['labels']).sum().item()
        total_correct += correct

    # Calculate average loss and accuracy for the training set
    accuracy = total_correct / total_samples * 100
    return total_loss / total_samples, accuracy

# Validation function
@torch.no_grad()  # Disable gradient calculations for validation
def val(loader):
    model.eval()  # Set the model to evaluation mode
    total_loss = total_samples = total_correct = 0
    y_true = []
    y_pred = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}  # Move to device (GPU/CPU)
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        total_loss += loss.item() * batch['input_ids'].size(0)  # Accumulate loss
        total_samples += batch['input_ids'].size(0)

        y_true.extend(batch['labels'].cpu().numpy())
        y_pred.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())

        # Calculate accuracy for this batch
        correct = (torch.argmax(outputs.logits, dim=-1) == batch['labels']).sum().item()
        total_correct += correct

    # Calculate average loss and accuracy for the validation set
    accuracy = total_correct / total_samples * 100
    return total_loss / total_samples, accuracy

In [ ]:
# Initialize lists to store training and validation losses and accuracies
training_loss = []
validation_loss = []
training_acc = []
validation_acc = []

# Training and validation loop
for epoch in range(epochs):
    # Train the model and get the training loss and accuracy
    train_loss, train_accuracy = train(train_loader)

    # Validate the model and get the validation loss and accuracy
    val_loss, val_accuracy = val(val_loader)

    # Append the results to the lists
    training_loss.append(train_loss)
    training_acc.append(train_accuracy)
    validation_loss.append(val_loss)
    validation_acc.append(val_accuracy)

    # Print progress for this epoch
    print(f"Epoch: {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

In [ ]:
# Choose a style for the plot
plt.style.use("default")

# Create figure
plt.figure(figsize=(10, 7))

# Plot training and validation loss
plt.plot(range(epochs), training_loss, label='Training Loss', 
         color='#ffda06', linestyle='-', linewidth=2.2, alpha=1)

plt.plot(range(epochs), validation_loss, label='Validation Loss', 
         color='red', linestyle='--', linewidth=2, alpha=1)

# Labels and title
plt.xlabel('Epochs', fontsize=14, fontweight='bold', color='#333333')
plt.ylabel('Loss', fontsize=14, fontweight='bold', color='#333333')
plt.title('Training and Validation Loss', fontsize=17, fontweight='bold', color='#222222')

plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)

plt.legend(loc='upper right', fontsize=13, frameon=False)

# Adjust layout for a clean look
plt.tight_layout()
plt.show()

In [ ]:
# Generate Submission
submission = pd.DataFrame()
submission['id'] = nlp_dataset['test']['id']

# Process test dataset
tokenized_datasets_test = tokenized_datasets_clean["test"]
tokenized_datasets_test = tokenized_datasets_test.remove_columns(["labels"])

test_dataloader = DataLoader(
    tokenized_datasets_test, batch_size=8, collate_fn=data_collator
)

predictions = []

# Switch model to evaluation mode
model.eval()

@torch.no_grad()
def make_predictions(batch):
    batch = {k: v.to(device) for k, v in batch.items()}  
    outputs = model(**batch)  
    logits = outputs.logits  
    return torch.argmax(logits, dim=-1)  

# Inference loop
for batch in test_dataloader:
    predictions.append(make_predictions(batch))

# Concatenate all predictions and move to CPU for further processing
all_preds = torch.cat(predictions).cpu().numpy()

# Prepare the submission dataframe
submission['target'] = all_preds

# Save the predictions to a CSV file
submission.to_csv('submission.csv', index=False)